# 국토부 버스정류장 API로 누락 정류장 조회

수요 데이터에 기록된 정류장 ID를 국토교통부 `BusStop/getBusStop` API로 조회해 정류장명, ARS 번호, 위도·경도를 확인합니다.

In [ ]:
from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'gtx_a_seoul_bus_outputs' / 'transport_card'
ENDPOINT = 'https://apis.data.go.kr/1613000/BusStop/getBusStop'
API_KEY = getpass('국토부 BusStop 일반 인증키를 입력하세요: ').strip()
DATE = '20241017'
if not API_KEY: raise ValueError('인증키가 입력되지 않았습니다.')

# 2024년 수요 데이터에서 정류장명 매핑이 비어 있는 ID를 자동 추출
input_fp = DATA_DIR / f'gtx_a_transport_card_{DATE}_raw_with_coords.csv'
demand = pd.read_csv(input_fp, dtype=str, encoding='utf-8-sig').fillna('')
missing = []
for _, row in demand.iterrows():
    for side, id_col, name_col, city_col in [('승차', 'ride_sttn_id', '승차정류장명', 'ride_ctpv_cd'), ('하차', 'goff_sttn_id', '하차정류장명', 'goff_ctpv_cd')]:
        if row[name_col].strip() == '' and row[id_col].strip() != '':
            missing.append({'구분': side, '노선번호': row.get('query_route_no', ''), '노선ID': row.get('query_route_id', ''), '정류장ID': row[id_col].strip(), '시도코드': row.get(city_col, '') or '41'})
missing = pd.DataFrame(missing).drop_duplicates()
targets = missing[['정류장ID', '시도코드']].drop_duplicates().reset_index(drop=True)
print(f'조회 대상: {len(targets)}개 정류장 ID')

def parse_items(payload):
    body = payload.get('response', payload)
    header = body.get('header', {})
    result = body.get('body', {}).get('items', {})
    items = result.get('item', []) if isinstance(result, dict) else result
    if isinstance(items, dict): items = [items]
    return items or [], header

records, logs = [], []
for i, row in targets.iterrows():
    params = {'serviceKey': API_KEY, 'pageNo': 1, 'numOfRows': 100, 'opr_ymd': DATE, 'sttn_id': row['정류장ID'], 'ctpv_cd': row['시도코드'], 'dataType': 'JSON'}
    try:
        response = requests.get(ENDPOINT, params=params, timeout=30)
        response.raise_for_status()
        items, header = parse_items(response.json())
        for item in items:
            item = dict(item); item['조회정류장ID'] = row['정류장ID']; item['조회시도코드'] = row['시도코드']; records.append(item)
        logs.append({'정류장ID': row['정류장ID'], '시도코드': row['시도코드'], '결과': header.get('resultCode', ''), '건수': len(items), '오류': ''})
        print(f'[{i+1}/{len(targets)}] {row["정류장ID"]}: {len(items)}건')
    except Exception as exc:
        logs.append({'정류장ID': row['정류장ID'], '시도코드': row['시도코드'], '결과': 'ERROR', '건수': 0, '오류': repr(exc)})
        print(f'[{i+1}/{len(targets)}] {row["정류장ID"]}: 실패 - {exc}')
    time.sleep(0.1)

result = pd.DataFrame(records)
log = pd.DataFrame(logs)
result.to_csv(DATA_DIR / f'missing_stop_api_result_{DATE}.csv', index=False, encoding='utf-8-sig')
log.to_csv(DATA_DIR / f'missing_stop_api_query_log_{DATE}.csv', index=False, encoding='utf-8-sig')
print(f'정류장 조회 결과 저장: {len(result)}건')
print(result.to_string(index=False) if not result.empty else '조회 결과 없음')